# DiCo-NLI Track 1 — Assignment 2 experiments (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells in order. Cell 2 asks you to upload three files from your Mac folder:
`src/decoding.py`, `src/train_a2.py`, `06_decoding_experiments.py`.

Models are trained in **fp32** (DeBERTa-v3 gives NaN in fp16 and on Apple MPS).

In [ ]:
# 1. Setup: clone the repo (data, official scorer, src/data.py)
!git clone -q https://github.com/MuneebBaig71/DiCo-NLI_SemEval.git
%cd DiCo-NLI_SemEval
!pip -q install sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Upload the three new A2 files and put them in place
import shutil
from google.colab import files
uploaded = files.upload()   # select decoding.py, train_a2.py, 06_decoding_experiments.py
dest = {"decoding.py": "src/decoding.py",
        "train_a2.py": "src/train_a2.py",
        "06_decoding_experiments.py": "06_decoding_experiments.py"}
for name in uploaded:
    shutil.move(name, dest.get(name, name))
    print("placed", name, "->", dest.get(name, name))
missing = [n for n in dest if n not in uploaded]
print("MISSING:", missing if missing else "none")

In [ ]:
# 3. (Recommended) save results to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/DiCo_A2_results"

In [ ]:
# 4. Smoke test (~2 min): 1 epoch of DeBERTa-v3 on 300 rows, with the consistency loss on.
#    Checks: no NaN, the loss terms print, files get written. Not a real result.
import importlib, src.decoding, src.train_a2
importlib.reload(src.decoding); importlib.reload(src.train_a2)
from src.train_a2 import run
meta = run(model="microsoft/deberta-v3-base", epochs=1, max_train_rows=300,
           lambda_cons=1.0, tag="smoke", out_dir="results_smoke")
print(meta["history"])

In [ ]:
# 5. Main runs, seed 42 (about 10 min each on a T4).
#    M0 = A1 RoBERTa rerun under the new protocol (fair comparison)
#    M1 = plain DeBERTa-v3, M2 = + consistency loss, M3 = M2 + Neg weighting
import json, shutil, pandas as pd
from pathlib import Path

EXPS = [
    dict(tag="M0_roberta",        model="roberta-base"),
    dict(tag="M1_deberta",        model="microsoft/deberta-v3-base"),
    dict(tag="M2_deberta_cons",   model="microsoft/deberta-v3-base", lambda_cons=1.0),
    dict(tag="M3_deberta_cons_neg", model="microsoft/deberta-v3-base", lambda_cons=1.0, neg_weight=2.0),
]
SEED = 42

for e in EXPS:
    tag = f"{e['tag']}_seed{SEED}"
    if Path(f"results_a2/runs/{tag}.json").exists():
        print("skip (done):", tag); continue
    run(**{**e, "tag": tag, "seed": SEED})
    try:   # copy after every run, so a disconnect loses at most one run
        shutil.copytree("results_a2", DRIVE_OUT, dirs_exist_ok=True)
    except NameError:
        pass

In [ ]:
# 6. Score every run on dev with the official scorer, under all 4 decoding rules
!python 06_decoding_experiments.py

In [ ]:
# 7. Training summary (best epoch chosen on the validation split, not dev)
rows = []
for f in sorted(Path("results_a2/runs").glob("*.json")):
    m = json.loads(f.read_text())
    rows.append(dict(tag=m["tag"], best_epoch=m["best_epoch"],
                     best_val_wf1=round(m["best_val_wf1"], 4), minutes=round(m["seconds"] / 60, 1)))
pd.DataFrame(rows)

In [ ]:
# 8. Download everything as a zip (unzip into your Mac folder)
!zip -qr results_a2.zip results_a2
files.download("results_a2.zip")